<a href="https://colab.research.google.com/github/okuya-net/double-loop-orchestrator/blob/main/DoubleLoop.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# <h1 style="font-size: 2.8em;">🌀 The Double-Loop Orchestrator</h1>
### **Stateless, Zero-Database GCS-Ledger Architecture**

---

## 💡 The Core Philosophy: Configuration as Data

In traditional software, changing logic requires a code deployment. In the **Double-Loop** architecture, we **decouple the 'Brain' (Logic) from the 'Body' (Execution)**. By storing your system's intelligence in **Google Cloud Storage (GCS)** as JSON 'Blueprints', you transform infrastructure into a dynamic, live-updating organism.

### **What lives in the Ledger?**
*   **🏗️ Pipeline Blueprints:** Sequence of steps including LLM calls and **Multimedia Processing** (e.g., Image Resizing, Video Compression).
*   **🧠 Model Metadata:** Assign LLM versions (e.g., Gemini 3.1 Pro vs. Flash-Lite) and support multi-model fallback logic.
*   **📜 System Instructions:** Specific prompts, persona definitions, and tone control.
*   **📐 Response Schemas:** Pydantic JSON structures to ensure the LLM responses will be written in the format you can immediately parse to your database.
*   **🧹 Lifecycle Rules:** Instructions for the orchestrator to **move, archive, or delete** processed files once finished.

---

## 🚀 Why this matters for Scaling & Future-Proofing

| Benefit | Description |
| :--- | :--- |
| **⚡ Zero Redeployments** | Upgrade models or change prompts by editing a JSON file. No CI/CD pipelines, no downtime. |
| **📊 Deterministic Output** | Update **Response Schemas** on the fly to support new UI features without refactoring the orchestrator. |
| **🛠️ Custom Processing** | Easily insert pre-processing (Image Resize) or post-processing (Compression) steps without refactoring core logic. |
| **🛡️ Future-Proofing** | The LLM landscape evolves weekly. Don't let your code rot; 'hot-swap' to newer models like Gemini 3.1 in seconds. |
| **📈 Infinite Scaling** | Stateless execution means you can process 1 file or 1 million simultaneously with no database bottlenecks. |

> **Showcase Goal:** Illustrate how a simple Python function can act as a world-class orchestrator when driven by a 'Configuration as Data' strategy.

In [ ]:
import os
import json

# --- 🏛️ INFRASTRUCTURE: THE STATELESS STORAGE ENGINE ---
# This class mimics the behavior of Google Cloud Storage.
# In a real environment, this would use the `google-cloud-storage` SDK.
# It ensures that our 'State' is never stored in RAM, but always persisted as data.

class GCSStorageClient:
    """A stateless storage abstraction for the Double-Loop Orchestrator."""
    def __init__(self, base_path="./mock_gcs_bucket"):
        self.base_path = base_path
        os.makedirs(base_path, exist_ok=True)

    def write_json(self, path, data):
        """Persistently saves configuration or execution state."""
        full_path = os.path.join(self.base_path, path)
        os.makedirs(os.path.dirname(full_path), exist_ok=True)
        with open(full_path, 'w') as f:
            json.dump(data, f, indent=2)
        return f"gs://mock-bucket/{path}"

    def read_json(self, path):
        """Retrieves the current 'Truth' from the storage ledger."""
        full_path = os.path.join(self.base_path, path)
        with open(full_path, 'r') as f:
            return json.load(f)

# Initialize our global storage engine
storage_client = GCSStorageClient()

### 🎙️ Multi-Step Use Case: Audio Intelligence (Gemini 3.1 Standard)
This example demonstrates a complex pipeline where we swap models and schemas dynamically.

1. **Step 1 (Transcription):** Uses `gemini-3.1-flash-lite` for ultra-fast text extraction.
2. **Step 2 (Sentiment):** Switches to `gemini-3.1-pro` for deep emotional intelligence and complex reasoning.
3. **Step 3 (Report):** Uses `gemini-3.1-flash` to summarize findings into a final markdown report.

**Note:** Because we define this in a JSON 'Blueprint', you could swap Step 2 to a different model just by editing the file—no code changes required!

In [ ]:
import os
import json

# 🌀 Use Case: Audio Intelligence (Gemini 3.1 Edition)
# This blueprint defines our models, instructions, and schemas externally.

audio_blueprint = {
    "pipeline_id": "audio_intel_v3",
    "version": "3.1.0",
    "stages": [
        {
            "id": "step_01_transcription",
            "executor": "vertex-speech-to-text",
            "parameters": {
                "model_name": "gemini-3.1-flash-lite", # Speedy initial processing
                "language": "en-US"
            }
        },
        {
            "id": "step_02_deep_sentiment",
            "executor": "vertex-reasoning-engine",
            "parameters": {
                "model_name": "gemini-3.1-pro", # Deep reasoning / high emotional intelligence
                "response_schema": {
                    "sentiment": "string",
                    "urgency_score": "integer",
                    "emotional_tone": "string"
                }
            }
        },
        {
            "id": "step_03_action_extraction",
            "executor": "vertex-report-generator",
            "parameters": {
                "model_name": "gemini-3.1-flash", # Balanced summarization
                "output_format": "markdown"
            }
        }
    ]
}

# Save the blueprint to our 'Stateless Brain'
storage_client.write_json("pipelines/audio_intel_v3.json", audio_blueprint)

print("✅ Multi-step Blueprint Created using Gemini 3.1!")
print("💡 Core Benefit: Swap models or logic in the JSON above to update the pipeline instantly.")

✅ Multi-step Blueprint Created using Gemini 3.1!
💡 Core Benefit: Swap models or logic in the JSON above to update the pipeline instantly.


## ⚡️ Deploying as a Production Cloud Function

Transitioning from this notebook to the cloud is seamless. In production, our code lives inside a **Google Cloud Function (2nd Gen)**.

### 🛠 The Trigger Mechanism
The function acts as the **Air Traffic Controller**. It wakes up only when needed:
*   📥 **New Data:** When a raw file is uploaded.
*   ✅ **Task Finished:** When a worker completes its job and saves its state.

In [ ]:
!pip install functions-framework --quiet

# --- 🚦 ROUTING: THE EVENT-DRIVEN ENTRY POINT ---
# This is the 'Double-Loop' router. It is stateless and idempotent.
# It doesn't care about previous sessions; it only looks at the GCS event data.

try:
    import functions_framework
    HAS_FF = True
except ImportError:
    HAS_FF = False
    print("Note: functions_framework not found. Using simulation mode.")

def orchestrator_logic(bucket, name):
    """Core routing logic shared between Cloud Function and local tests."""
    print(f"Processing: {name} in {bucket}")
    # 1. Is this a new upload? Start a new Ledger.
    # 2. Is this a worker finishing? Update the Ledger.
    # 3. Is the pipeline complete? Run Lifecycle Rules.
    return "Success"

if HAS_FF:
    @functions_framework.cloud_event
    def gcs_orchestrator_router(cloud_event):
        data = cloud_event.data
        orchestrator_logic(data['bucket'], data['name'])
        return "OK", 200

ModuleNotFoundError: No module named 'functions_framework'

### 3. Testing the Deployment
You can test the 'Stateless' nature of the orchestrator by simulating a GCS event. Since the state is stored in the JSON ledger, you don't need a running server—only the GCS file.

In [ ]:
# Mock test script to simulate a Cloud Function trigger
def test_cloud_function_trigger(simulated_file_path):
    console.print(f"[bold yellow]Testing Orchestrator Trigger for: {simulated_file_path}[/bold yellow]")

    # Simulate the router logic
    if "raw_files/" in simulated_file_path:
        new_id = orchestrator.create_execution("document_ingestion_v1", f"gs://{BUCKET_NAME}/{simulated_file_path}")
        console.print(f"[green]Success:[/green] New execution ledger created: {new_id}")

    # Run the next stage immediately (The 'Loop')
    next_task = orchestrator.run_next_stage(new_id)
    console.print(f"[blue]Dispatcher:[/blue] Launching worker for {next_task['executor']}")

# Run the test
test_cloud_function_trigger("raw_files/new_invoice_test.pdf")

## 🚀 How to Deploy to Google Cloud

To move this from a notebook to production, you would follow these steps using the Google Cloud CLI (`gcloud`). This command creates a **2nd Gen Cloud Function** that automatically wakes up whenever a file is created in your bucket.

### Deployment Command
Run this in your terminal to deploy the `gcs_orchestrator_router` function defined above:

In [ ]:
# @title Deployment Shell Script (Template)
# This cell contains the shell command for deployment.
# Note: This is for demonstration; execution requires a configured gcloud environment.

deploy_command = """
gcloud functions deploy gcs-orchestrator-router \
    --gen2 \
    --runtime=python311 \
    --region=us-central1 \
    --source=./source_code_dir \
    --entry-point=gcs_orchestrator_router \
    --trigger-event-filters="type=google.cloud.storage.object.v1.finalized" \
    --trigger-event-filters="bucket=YOUR_GCS_BUCKET_NAME" \
    --service-account=your-service-account@your-project.iam.gserviceaccount.com
"""

console.print(Panel(deploy_command, title="gcloud Deployment Command", border_style="green"))

### What this command does:
1.  **`--gen2`**: Uses the latest Cloud Functions architecture (Cloud Run based).
2.  **`--trigger-event-filters`**: This is the magic. It tells Google Cloud to watch for the `finalized` event (new file created) in your specific bucket.
3.  **Zero Maintenance**: Once deployed, you never have to manage a server. If 1,000 files are uploaded at once, Google will scale up 1,000 instances of this function, then scale back down to zero when finished.

```markdown
## 🏁 Conclusion: The Stateless Advantage

By following this **Double-Loop** pattern, you’ve built a system that is:

*   **🛠️ Maintenance-Free:** No databases to patch or scale.
*   **🎥 Multimedia Ready:** Beyond LLMs, this architecture handles complex pipelines including **Image Resizing** and **File Compression** by simply adding those steps to the Blueprint.
*   **📐 Schema-Driven:** Control your data contracts via **Response Schemas** stored in GCS, making your AI outputs reliable and predictable.
*   **🧹 Self-Cleaning:** The orchestrator can automatically **delete or move** files to 'Processed' or 'Archive' folders, keeping your production buckets organized.
*   **📈 Infinitely Scalable:** Handles 1 or 1,000,000 requests with the same logic.
*   **⚡ Instant Updates:** Change your AI's behavior, model version, or output schema by simply editing a JSON file in GCS.

**Next Steps:** To move to production, simply replace the `GCSStorageClient` mock with the official `google-cloud-storage` library and deploy using the provided `gcloud` command!
```

## 🚀 Scaling to the World: GitHub & Colab Workflow

To share this as a professional showcase, use **GitHub** as your documentation home and **Colab** as your interactive playground. This allows users to test your 'Double-Loop' architecture with a single click.

### 1. The 'Open in Colab' Button
Add this Markdown to your GitHub `README.md`. It creates a button that instantly clones your repo into a live Colab session:

```markdown
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR_USERNAME/YOUR_REPO_NAME/blob/main/Double_Loop_Orchestrator.ipynb)
```

### 2. Connecting to Real GCS (Sandbox Mode Switch)
In the 'Playground' phase, users need to authenticate. Run the following cell to switch from the local simulation to a real Google Cloud environment.

In [ ]:
# @title 🛡️ Authenticate to Google Cloud
from google.colab import auth

# This launches the standard Google login flow.
# It uses Application Default Credentials (ADC) to safely grant the notebook
# temporary access to your GCS buckets and Gemini models.
auth.authenticate_user()

print("✅ Authenticated! You can now toggle USE_REAL_GCS = True in the infrastructure cell.")

### 3. Folder Structure for GitHub
Organize your repository like this to keep the 'Configuration as Data' philosophy clean:

```text
├── README.md               <-- Documentation & Colab Badge
├── Double_Loop.ipynb       <-- This notebook
├── blueprints/             <-- Example JSON configs
│   ├── audio_pipeline.json
│   └── image_resize.json
└── cloud_functions/        <-- Production main.py & requirements.txt
    ├── main.py             
    └── requirements.txt
```